# QLoRA fine-tune: SOC triage agent on Qwen2.5-7B-Instruct

Runs `training/train_qlora.py` on a free Colab T4 or Kaggle GPU. Before you start:

1. On a machine with the trace store, run `python -m training.build_sft_data` and upload
   `training/data/sft_train.jsonl` and `sft_val.jsonl` (to this runtime, or to Google Drive).
2. Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Repo + training extras (Unsloth pins compatible torch/transformers/trl versions).
!git clone https://github.com/jeff13in/Agentic-Soc.git
%cd Agentic-Soc
!pip install -q unsloth
!pip install -q -e . --no-deps
!pip install -q pydantic-settings duckdb pyyaml

Put the SFT files in `training/data/`. On Colab you can upload them with the file browser, or
mount Drive and copy them:

In [ ]:
import os, shutil
os.makedirs("training/data", exist_ok=True)
# from google.colab import drive; drive.mount("/content/drive")
# for f in ("sft_train.jsonl", "sft_val.jsonl"):
#     shutil.copy(f"/content/drive/MyDrive/soc-triage/{f}", "training/data/")
print(os.listdir("training/data"))

Train. LoRA rank 16, alpha 32, 2 epochs, loss on assistant turns only. On a T4 expect
roughly 1-2 minutes per 100 examples per epoch at 12k context; lower `--max-seq-length` if you
run out of memory.

In [ ]:
!python -m training.train_qlora --data training/data --out training/out/soc-qwen-lora \
    --epochs 2 --rank 16 --alpha 32 --max-seq-length 12288 --report-to tensorboard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir training/out/soc-qwen-lora/checkpoints

Save the adapter. Pushing to the Hugging Face Hub publishes it (log in first with a write
token); otherwise download the zip.

In [ ]:
!cat training/out/soc-qwen-lora/training_meta.json
!cd training/out/soc-qwen-lora && zip -qr adapter.zip adapter && ls -lh adapter.zip
# from huggingface_hub import login; login()
# !python -m training.train_qlora ... --push-to-hub <your-user>/soc-qwen2.5-7b-triage-lora

Next: serve it with `training/serve_vllm.sh` on a GPU box and run
`python -m evals.benchmark` from the repo (see the README's Phase 3 section).